## 4. Production Pipeline: Full Preprocessing with ColumnTransformer

In real-world projects, we build a single unified preprocessing pipeline:
1. **Nominal Features** $\to$ `OneHotEncoder(handle_unknown='ignore')`
2. **Ordinal Features** $\to$ `OrdinalEncoder(categories=[...])`
3. **High-Cardinality Features** $\to$ `TargetEncoder(cv=3, smooth='auto')`
4. **Numerical Features** $\to$ `StandardScaler()` (or `RobustScaler()` if extreme outliers exist)
5. **No Data Leakage** $\to$ `fit_transform` on `X_train`, `transform` on `X_test`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# 1. Realistic enterprise dataset with mixed feature types
data = {
    'Age': [25, 34, 45, 22, 28, 52, 40, 29, 36, 48, 23, 41],
    'Annual_Income': [35000, 78000, 120000, 28000, 65000, 140000, 95000, 52000, 88000, 110000, 31000, 99000],
    'Credit_Score': [610, 720, 790, 580, 690, 820, 750, 640, 710, 800, 600, 740],
    'Device_Type': ['Android', 'iOS', 'Android', 'Windows', 'iOS', 'iOS', 'Android', 'Windows', 'iOS', 'Android', 'Android', 'iOS'],
    'Education_Level': ['High School', 'Bachelors', 'PhD', 'High School', 'Masters', 'PhD', 'Bachelors', 'Masters', 'Bachelors', 'PhD', 'High School', 'Masters'],
    'City_Pincode': ['500001', '560001', '400001', '500001', '560001', '110001', '500001', '400001', '560001', '600001', '500001', '110001'],
    'Loan_Approved': [0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1]  # Target (y)
}

df = pd.DataFrame(data)
print("=== RAW INPUT DATASET ===")
display(df.head())

=== RAW INPUT DATASET ===


,Age,Annual_Income,Credit_Score,Device_Type,Education_Level,City_Pincode,Loan_Approved
0,25,35000,610,Android,High School,500001,0
1,34,78000,720,iOS,Bachelors,560001,1
2,45,120000,790,Android,PhD,400001,1
3,22,28000,580,Windows,High School,500001,0
4,28,65000,690,iOS,Masters,560001,1


---
## Step 1: Train-Test Split (Mandatory First Step)
Always split your data before fitting any encoders or scalers to prevent data leakage.

In [2]:
X = df.drop(columns=['Loan_Approved']).copy()
y = df['Loan_Approved'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

Training rows: 9 | Test rows: 3


---
## Step 2: Build the Unified `ColumnTransformer`

Group columns by their preprocessing strategy:
* `num_cols`: Scaled with `StandardScaler()`
* `nominal_cols`: Encoded with `OneHotEncoder()`
* `ordinal_cols`: Encoded with `OrdinalEncoder()`
* `high_card_cols`: Encoded with `TargetEncoder()`

In [3]:
# Define column subsets
num_columns = ['Age', 'Annual_Income', 'Credit_Score']

# Assemble the complete preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num_scaler', StandardScaler(), num_columns),
        ('nominal_ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['Device_Type']),
        ('ordinal_enc', OrdinalEncoder(categories=[['High School','Bachelors', 'PhD', 'Masters']]), ['Education_Level']),
        ('target_enc', TargetEncoder(cv=3, smooth='auto', random_state=42), ['City_Pincode'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')
X_train_transformed = preprocessor.fit_transform(X_train, y_train)

X_test_transformed = preprocessor.transform(X_test)

print("=== TRANSFORMED TRAINING FEATURES (ENCODED & SCALED) ===")
display(X_train_transformed.head())

=== TRANSFORMED TRAINING FEATURES (ENCODED & SCALED) ===


d:\machine_learning_code\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


,Age,Annual_Income,Credit_Score,Device_Type_Android,Device_Type_Windows,Device_Type_iOS,Education_Level,City_Pincode
8,-0.038069,0.093116,-0.080353,0.0,0.0,1.0,1.0,1.000000
5,1.789259,1.707120,1.510637,0.0,0.0,1.0,2.0,1.000000
2,0.989803,1.086349,1.076731,1.0,0.0,0.0,2.0,0.833333
1,-0.266485,-0.217270,0.064282,0.0,0.0,1.0,1.0,1.000000
11,0.532971,0.434540,0.353553,0.0,0.0,1.0,3.0,1.000000


---
## Step 3: Bundle into an End-to-End Pipeline & Train

By bundling `preprocessor` + `model` in a `Pipeline`:
* You call `.fit(X_train, y_train)` once.
* When evaluating or serving via an API, `.predict(X_test)` takes raw unprocessed JSON/DataFrame and does all scaling/encoding automatically under the hood.

In [ ]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)

print("=== MODEL PERFORMANCE ON TEST DATA ===")
print(classification_report(y_test, y_pred))

=== MODEL PERFORMANCE ON TEST DATA ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



d:\machine_learning_code\.venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(


---
## Summary of Production Preprocessing Rules

1. **Group by Data Type**: Identify numeric, nominal, ordinal, and high-cardinality features up front.
2. **Select Scaler by Algorithm**:
   * Tree Models $\to$ Skip scaling (`passthrough`).
   * Linear / Distance / Neural Models $\to$ `StandardScaler()`.
3. **Fit on Train Only**: Never call `.fit()` or `.fit_transform()` on `X_test` or unseen production data.
4. **Single Pipeline Artifact**: Wrap everything in `Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])` for seamless deployment.